# Part 03 — Tabular ML Training & Ethics

Satellite notebook split from `capstone_with_results.ipynb` to keep the master lighter to open.
Outputs/source sync automatically via `scripts/sync_notebook_parts.py` (run the final sync cell).


In [ ]:
# ── PART BOOTSTRAP: Part 03 — Tabular ML Training & Ethics ──
# Lightweight satellite notebook — syncs back into notebooks/capstone_with_results.ipynb
import sys
from pathlib import Path

def _find_root() -> Path:
    for base in [Path.cwd(), *Path.cwd().parents]:
        if (base / "src" / "vision_brain.py").is_file() and (base / "explanations" / "build_all.py").is_file():
            return base
        if (base / "src" / "vision_brain.py").is_file():
            return base
    return Path.cwd().parent if Path.cwd().name in {"notebooks", "parts"} else Path.cwd()

REPO_ROOT = _find_root()
SRC = REPO_ROOT / "src"
DATA = REPO_ROOT / "Data"
if not DATA.is_dir():
    DATA = REPO_ROOT / "data"
sys.path.insert(0, str(SRC))
sys.path.insert(0, str(REPO_ROOT / "scripts")) if (REPO_ROOT / "scripts").is_dir() else None

try:
    import torch
    TORCH_OK = True
except Exception:
    TORCH_OK = False

# Optional shared helpers used by several sections
try:
    from cm_helpers import plot_confusion_matrices_pair  # noqa: F401
except Exception:
    plot_confusion_matrices_pair = None

import numpy as np  # noqa: F401
import pandas as pd  # noqa: F401
import matplotlib.pyplot as plt  # noqa: F401

print(f"Part notebook ready | root={REPO_ROOT} | TORCH_OK={TORCH_OK} | DATA={DATA}")
print("When finished: run the last cell to sync outputs → capstone_with_results.ipynb")

---

## Section 8 · Model Training & Evaluation Pipeline

### Pipeline overview

| Step | What | Science |
|---|---|---|
| 8.1 | Data prep + SMOTE | Balances rare Fatal class without data leakage |
| 8.2 | 5 baseline classifiers | Honest performance floor; no tuning |
| 8.3 | Dynamic GridSearchCV | Exhaustive hyperparameter search with stratified 5-fold CV |
| 8.4 | PyTorch DNN | Matches Jiang et al. (2024) architecture: 256→128→64→3 |
| 8.5 | Head-to-head comparison | All models ranked by Macro Recall, MCC, AUC, F1, Accuracy |
| 8.6 | Ontario live test cases | 5 realistic highway scenarios to stress-test the winner |
| 8.7 | Final selection | Quantitative + qualitative rationale; Safety Score integration |

> **Primary metric: Macro Recall** — a missed Fatal prediction is far costlier than a false alarm.

### Section 8.1 · Data Preparation for Modelling

SMOTE is applied to the **training set only** (never the test set) to avoid
data leakage. The test set must reflect the real-world class distribution.

In [35]:
# ── 8.1 Data preparation (self-loading if session is fresh) ──────────────────
import os as _os

# Auto-load df_toronto if not in session (e.g. jumped directly to Section 8)
if "df_toronto" not in dir() or not isinstance(df_toronto, pd.DataFrame):
    print("df_toronto not found – auto-loading...")
    _nb_dir  = Path(_os.path.abspath(""))
    _DATA    = next((p for p in [_nb_dir/"Data", _nb_dir.parent/"Data"]
                     if p.is_dir()), _nb_dir/"Data")
    for _name in ["traffic collision data.csv","traffic_collision_data.csv"]:
        _p = _DATA / _name
        if _p.exists():
            df_toronto = pd.read_csv(_p, low_memory=False)
            print(f"  Loaded: {_name}  ({len(df_toronto):,} rows)")
            break
    else:
        raise FileNotFoundError(f"Toronto CSV not found in {_DATA}. Run Cell 5 first.")

# Ensure SEVERITY exists
if "SEVERITY" not in df_toronto.columns:
    df_toronto["SEVERITY"] = df_toronto.apply(
        lambda r: 2 if (pd.notnull(r.get("FATALITIES")) and r["FATALITIES"] > 0)
                  else (1 if str(r.get("INJURY_COLLISIONS","")).upper() == "YES" else 0),
        axis=1)

# Ensure all model features exist
FINAL_FEATURES = ["OCC_HOUR","MONTH_NUM","SEASON_NUM",
                   "IS_NIGHT","IS_RUSHHOUR",
                   "PEDESTRIAN_BIN","BICYCLE_BIN","AUTOMOBILE_BIN"]
_MONTH_MAP  = {m: i+1 for i, m in enumerate(
    ["January","February","March","April","May","June",
     "July","August","September","October","November","December"])}
_SEASON_MAP = {1:1,2:1,3:2,4:2,5:2,6:3,7:3,8:3,9:4,10:4,11:4,12:1}

if "MONTH_NUM" not in df_toronto.columns:
    df_toronto["MONTH_NUM"]  = df_toronto["OCC_MONTH"].map(_MONTH_MAP).fillna(6).astype(int)
if "SEASON_NUM" not in df_toronto.columns:
    df_toronto["SEASON_NUM"] = df_toronto["MONTH_NUM"].map(_SEASON_MAP)
_hr_col = next((c for c in ["OCC_HOUR","HOUR"] if c in df_toronto.columns), None)
if "OCC_HOUR" not in df_toronto.columns and _hr_col:
    df_toronto["OCC_HOUR"] = pd.to_numeric(df_toronto[_hr_col], errors="coerce").fillna(12).astype(int)
if "IS_NIGHT" not in df_toronto.columns:
    df_toronto["IS_NIGHT"]    = df_toronto["OCC_HOUR"].apply(lambda h: 1 if h<6 or h>=22 else 0)
if "IS_RUSHHOUR" not in df_toronto.columns:
    df_toronto["IS_RUSHHOUR"] = df_toronto["OCC_HOUR"].apply(lambda h: 1 if (7<=h<=9) or (16<=h<=18) else 0)
for _bin, _src in [("PEDESTRIAN_BIN","PEDESTRIAN"),("BICYCLE_BIN","BICYCLE"),("AUTOMOBILE_BIN","AUTOMOBILE")]:
    if _bin not in df_toronto.columns:
        df_toronto[_bin] = (df_toronto.get(_src, pd.Series(["No"]*len(df_toronto)))
                             .astype(str).str.upper().str.strip() == "YES").astype(int)

available   = [f for f in FINAL_FEATURES if f in df_toronto.columns]
df_model_m8 = df_toronto[available + ["SEVERITY"]].dropna().copy()
X = df_model_m8[available].values
y = df_model_m8["SEVERITY"].values

print(f"Feature matrix : {X.shape}")
print(f"Features       : {available}")
print("Class distribution:")
for cls, cnt in zip(*np.unique(y, return_counts=True)):
    print(f"  Class {cls}: {cnt:,}  ({cnt/len(y)*100:.1f}%)")

# Stratified split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y)
print(f"\nTrain: {X_train.shape[0]:,}   Test: {X_test.shape[0]:,}")

# SMOTE
if SMOTE_OK:
    sm = SMOTE(random_state=42, k_neighbors=3)
    X_train_sm, y_train_sm = sm.fit_resample(X_train, y_train)
    print(f"After SMOTE: {X_train_sm.shape[0]:,} samples")
else:
    X_train_sm, y_train_sm = X_train, y_train
    print("SMOTE skipped — using class_weight='balanced'")

# Scale
scaler     = StandardScaler()
X_train_sc = scaler.fit_transform(X_train_sm)
X_test_sc  = scaler.transform(X_test)
print("Scaling done.")

Feature matrix : (768000, 8)
Features       : ['OCC_HOUR', 'MONTH_NUM', 'SEASON_NUM', 'IS_NIGHT', 'IS_RUSHHOUR', 'PEDESTRIAN_BIN', 'BICYCLE_BIN', 'AUTOMOBILE_BIN']
Class distribution:
  Class 0: 663,267  (86.4%)
  Class 1: 104,089  (13.6%)
  Class 2: 644  (0.1%)

Train: 614,400   Test: 153,600
After SMOTE: 1,591,842 samples
Scaling done.


### Section 8.2 · Baseline Models

Five classifiers trained with **default settings** to establish the performance floor.

| Model | Inductive bias | Why included |
|---|---|---|
| Logistic Regression | Linear boundaries | Project deliverable (interpretable coefficients) |
| Decision Tree | Axis-aligned splits | Fast; shows best splitting features |
| K-Nearest Neighbours | Local manifold | No distributional assumption |
| Random Forest | Random tree ensemble | Paper 2 benchmark target (87.8% acc) |
| LightGBM | Gradient boosting | State-of-the-art on tabular data |

In [ ]:
# ── 8.2  Baseline Models ──────────────────────────────────────────────────────
# SPEED OPTIMISATIONS vs original:
#   • LogisticRegression  n_jobs=-1   (parallelised over classes)
#   • RandomForest        n_jobs=-1   (trees built in parallel)
#   • LightGBM            n_jobs=-1
#   • KNN                 n_jobs=-1, algorithm='ball_tree'  (much faster on 600k rows)
#   • All fitted on the SMOTE-balanced training set (X_train_sc, y_train_sm)

import time

def evaluate(name, model, X_tr, y_tr, X_te, y_te):
    t0 = time.time()
    model.fit(X_tr, y_tr)
    preds = model.predict(X_te)
    proba = model.predict_proba(X_te) if hasattr(model, "predict_proba") else None
    auc = (round(roc_auc_score(y_te, proba, multi_class="ovr", average="macro"), 4)
           if proba is not None else None)
    return {
        "Model"    : name,
        "Accuracy" : round(accuracy_score(y_te, preds), 4),
        "Prec (M)" : round(precision_score(y_te, preds, average="macro",    zero_division=0), 4),
        "Rec (M)"  : round(recall_score(   y_te, preds, average="macro",    zero_division=0), 4),
        "F1 (M)"   : round(f1_score(       y_te, preds, average="macro",    zero_division=0), 4),
        "F1 (W)"   : round(f1_score(       y_te, preds, average="weighted", zero_division=0), 4),
        "MCC"      : round(matthews_corrcoef(y_te, preds), 4),
        "AUC (OvR)": auc,
        "Time (s)" : round(time.time() - t0, 1),
        "_model"   : model,
        "_preds"   : preds,
    }

baseline_models = [
    ("Logistic Regression",
     LogisticRegression(max_iter=1000, class_weight="balanced",
                        n_jobs=-1, solver="lbfgs")),

    ("Decision Tree",
     DecisionTreeClassifier(max_depth=15, class_weight="balanced",
                            random_state=42)),

    ("K-Nearest Neighbours",
     # ball_tree + n_jobs=-1 is critical on 600k rows; k=7 is a good default
     KNeighborsClassifier(n_neighbors=7, algorithm="ball_tree",
                          n_jobs=-1, leaf_size=40)),

    ("Random Forest",
     RandomForestClassifier(n_estimators=100, class_weight="balanced",
                            n_jobs=-1, random_state=42)),
]
if LGBM_OK:
    baseline_models.append(
        ("LightGBM",
         lgb.LGBMClassifier(n_estimators=200, class_weight="balanced",
                            n_jobs=-1, random_state=42, verbose=-1))
    )

baseline_results = []
for name, model in baseline_models:
    print(f"  Training {name} ...", end=" ", flush=True)
    r = evaluate(name, model, X_train_sc, y_train_sm, X_test_sc, y_test)
    baseline_results.append(r)
    print(f"done in {r['Time (s)']}s  |  F1(M)={r['F1 (M)']}  Acc={r['Accuracy']}")

# ── Pretty table ──────────────────────────────────────────────────────────────
cols = ["Model","Accuracy","Prec (M)","Rec (M)","F1 (M)","F1 (W)","MCC","AUC (OvR)","Time (s)"]
df_bl = pd.DataFrame(baseline_results)[cols]
print("\n=== Baseline Model Results ===")
print(df_bl.to_string(index=False))


  Training Logistic Regression ... done in 6.6s  |  F1(M)=0.3507  Acc=0.7779
  Training Decision Tree ... done in 8.0s  |  F1(M)=0.3805  Acc=0.7607
  Training K-Nearest Neighbours ... 

---

## Section 7 · AI Ethics & Fairness Audit

> **Why this matters**: Ontario Smart-Shield directly influences speed recommendations
> on public highways. A biased model that performs well on average but poorly for
> specific groups or geographies could increase — not decrease — road risk for
> vulnerable populations. This section documents and measures four ethical obligations.

---

### 7.1  Ethical Risk Register

| Risk | Category | Likelihood | Impact | Mitigation in this project |
|---|---|---|---|---|
| Model ignores rare Fatal class | **Class bias** | High | Critical | SMOTE oversampling + class_weight="balanced" + Recall KPI ≥ 92 % |
| Better performance in high-density urban areas | **Geographic bias** | Medium | High | Subgroup audit: urban vs. suburban/rural divisions |
| Night-time collisions under-represented | **Temporal bias** | Medium | Medium | `is_night` engineered feature; hour-of-day included |
| Model reflects historical policing patterns | **Systemic bias** | Medium | High | Disclosed; future work: fairness-aware reweighting |
| Opaque black-box recommendations | **Explainability** | Low-Med | Medium | Feature importances + SHAP values (Sprint 3) |

---

### 7.2  Fairness Metrics Defined

We measure **three complementary fairness criteria**:

| Criterion | Formula | Pass Threshold |
|---|---|---|
| **Equal Opportunity** (Fatal recall) | TP_fatal / (TP_fatal + FN_fatal) | ≥ 0.92 |
| **Demographic Parity** (per geography) | |Acc_urban − Acc_rural| | ≤ 0.05 |
| **Class Imbalance Ratio** | n_majority / n_minority | Disclosed; SMOTE applied |

---

### 7.3  Code: Class Imbalance Disclosure + Per-Class Recall + Geographic Audit


In [ ]:
# ── 7.3  AI Ethics & Fairness Audit ─────────────────────────────────────────
# Runs AFTER Section 8.2 baseline models have been trained.
# Uses the best baseline Random Forest (or best available) for the subgroup audit.

# ── 7.3.1  Class imbalance disclosure ────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

labels  = ["PD-Only (0)", "Injury (1)", "Fatal (2)"]
raw_cnt = np.bincount(y_test, minlength=3)
sm_cnt  = np.bincount(y_train_sm, minlength=3)

axes[0].bar(labels, raw_cnt, color=["#4C72B0","#DD8452","#C44E52"], edgecolor="white")
axes[0].set_title("Test-Set Class Distribution (raw)", fontweight="bold")
axes[0].set_ylabel("Count")
for ax_bar, v in zip(axes[0].patches, raw_cnt):
    axes[0].text(ax_bar.get_x()+ax_bar.get_width()/2, v+200,
                 f"{v:,}\n({v/raw_cnt.sum()*100:.1f}%)", ha="center", fontsize=9)

axes[1].bar(labels, sm_cnt, color=["#4C72B0","#DD8452","#C44E52"], edgecolor="white")
axes[1].set_title("After SMOTE – Training Set Distribution", fontweight="bold")
axes[1].set_ylabel("Count")
for ax_bar, v in zip(axes[1].patches, sm_cnt):
    axes[1].text(ax_bar.get_x()+ax_bar.get_width()/2, v+200,
                 f"{v:,}", ha="center", fontsize=9)

plt.suptitle("Fairness Audit 1 · Class Imbalance Disclosure", fontsize=13, fontweight="bold")
plt.tight_layout(); plt.show()
print(f"Imbalance ratio (raw): {raw_cnt.max()/max(raw_cnt.min(),1):.0f}:1  "
      f"→ SMOTE compressed to {sm_cnt.max()/max(sm_cnt.min(),1):.1f}:1")

# ── 7.3.2  Per-class recall audit (Equal Opportunity) ────────────────────────
print("\n=== Fairness Audit 2 · Per-Class Recall (Equal Opportunity) ===")
KPI_FATAL_RECALL = 0.92

audit_rows = []
for r in baseline_results:
    preds = r["_preds"]
    rc = recall_score(y_test, preds, average=None, zero_division=0)
    audit_rows.append({
        "Model"         : r["Model"],
        "Recall PD-Only": round(rc[0], 4),
        "Recall Injury" : round(rc[1], 4),
        "Recall Fatal"  : round(rc[2], 4),
        "KPI Met (≥0.92)": "YES" if rc[2] >= KPI_FATAL_RECALL else "NO ✗",
    })

df_audit = pd.DataFrame(audit_rows)
print(df_audit.to_string(index=False))

fig, ax = plt.subplots(figsize=(10, 4))
x = np.arange(len(df_audit))
w = 0.25
ax.bar(x - w, df_audit["Recall PD-Only"], w, label="PD-Only", color="#4C72B0")
ax.bar(x,     df_audit["Recall Injury"],  w, label="Injury",  color="#DD8452")
ax.bar(x + w, df_audit["Recall Fatal"],   w, label="Fatal",   color="#C44E52")
ax.axhline(KPI_FATAL_RECALL, color="red", linestyle="--", linewidth=1.5,
           label=f"Fatal Recall KPI ({KPI_FATAL_RECALL})")
ax.set_xticks(x); ax.set_xticklabels(df_audit["Model"], rotation=20, ha="right")
ax.set_ylabel("Recall"); ax.set_ylim(0, 1.05)
ax.set_title("Fairness Audit 2 · Per-Class Recall by Model", fontweight="bold")
ax.legend(); plt.tight_layout(); plt.show()

# ── 7.3.3  Geographic subgroup audit (Demographic Parity) ────────────────────
print("\n=== Fairness Audit 3 · Geographic Subgroup (Urban vs. Suburban/Rural) ===")

# Identify the best baseline RF result
_rf_result = next((r for r in baseline_results if "Random Forest" in r["Model"]), None)
if _rf_result is None:
    print("Random Forest baseline not found — skipping geographic audit.")
else:
    _model = _rf_result["_model"]

    # Build a local copy of the test slice of df_toronto with DIVISION info
    try:
        # Re-engineer the same rows used in Section 8.1 test split
        # (We align by resetting index consistently with Section 8.1 preprocessing)
        _dft2 = df_toronto.copy()
        _dft2["SEVERITY"] = _dft2.apply(
            lambda row: 2 if pd.notnull(row.get("FATALITIES")) and row["FATALITIES"] > 0
                        else (1 if str(row.get("INJURY_COLLISIONS","")).upper() == "YES" else 0),
            axis=1)
        bool_cols = ["AUTOMOBILE","MOTORCYCLE","PASSENGER","BICYCLE","PEDESTRIAN"]
        for c in bool_cols:
            _dft2[c] = (_dft2[c].astype(str).str.upper() == "YES").astype(int)
        _dft2["hour"]     = pd.to_numeric(_dft2.get("OCC_HOUR", 0), errors="coerce").fillna(0).astype(int)
        _dft2["is_night"] = ((_dft2["hour"] < 6) | (_dft2["hour"] >= 20)).astype(int)
        _dft2["month"]    = pd.to_datetime(_dft2.get("OCC_DATE",""), errors="coerce").dt.month.fillna(0).astype(int)
        _dft2["season"]   = _dft2["month"].map(
            {12:3,1:3,2:3,3:0,4:0,5:0,6:1,7:1,8:1,9:2,10:2,11:2}).fillna(0).astype(int)
        _dft2["dow"]      = pd.to_datetime(_dft2.get("OCC_DATE",""), errors="coerce").dt.dayofweek.fillna(0).astype(int)

        FEAT_COLS = ["hour","is_night","month","season","dow",
                     "AUTOMOBILE","MOTORCYCLE","PASSENGER","BICYCLE","PEDESTRIAN"]
        feat_ok = [c for c in FEAT_COLS if c in _dft2.columns]
        _dft2 = _dft2.dropna(subset=feat_ok + ["SEVERITY"])
        _dft2 = _dft2.reset_index(drop=True)

        # Urban = divisions D11-D55 (Toronto core); others are suburban/rural
        if "DIVISION" in _dft2.columns:
            urban_divs  = {"D11","D12","D13","D14","D22","D23","D31","D32",
                           "D33","D41","D42","D43","D51","D52","D53","D55"}
            _dft2["geo"] = _dft2["DIVISION"].apply(
                lambda d: "Urban" if str(d).strip().upper() in urban_divs else "Suburban/Rural")
        else:
            # Fall back to lat/lon centroid split if DIVISION missing
            _dft2["geo"] = np.where(
                (_dft2.get("LAT_WGS84", 43.7) > 43.65) &
                (_dft2.get("LONG_WGS84", -79.4).between(-79.5, -79.3)),
                "Urban", "Suburban/Rural")

        # Sample 50k for speed
        _sample = _dft2.sample(min(50000, len(_dft2)), random_state=42)
        X_geo   = StandardScaler().fit_transform(_sample[feat_ok].values)
        y_geo   = _sample["SEVERITY"].values
        geo_lbl = _sample["geo"].values

        geo_results = []
        for geo in ["Urban", "Suburban/Rural"]:
            mask = geo_lbl == geo
            if mask.sum() < 100:
                continue
            preds_geo = _model.predict(X_geo[mask])
            geo_results.append({
                "Geography"  : geo,
                "N"          : int(mask.sum()),
                "Accuracy"   : round(accuracy_score(y_geo[mask], preds_geo), 4),
                "F1 (Macro)" : round(f1_score(y_geo[mask], preds_geo, average="macro", zero_division=0), 4),
                "Recall Fatal": round(recall_score(y_geo[mask], preds_geo, average=None, zero_division=0)[2]
                                      if len(np.unique(y_geo[mask])) == 3 else 0, 4),
            })

        df_geo = pd.DataFrame(geo_results)
        print(df_geo.to_string(index=False))

        if len(df_geo) == 2:
            gap = abs(df_geo["Accuracy"].iloc[0] - df_geo["Accuracy"].iloc[1])
            print(f"\nDemographic Parity Gap (Accuracy): {gap:.4f}  "
                  f"({'PASS ≤0.05' if gap <= 0.05 else 'FAIL >0.05 — review needed'})")

            fig, ax = plt.subplots(figsize=(8, 4))
            mets = ["Accuracy","F1 (Macro)","Recall Fatal"]
            x    = np.arange(len(mets))
            ax.bar(x - 0.2, df_geo.iloc[0][mets], 0.4, label="Urban",          color="#4C72B0")
            ax.bar(x + 0.2, df_geo.iloc[1][mets], 0.4, label="Suburban/Rural", color="#55A868")
            ax.set_xticks(x); ax.set_xticklabels(mets)
            ax.axhline(0.92, color="red", linestyle="--", linewidth=1,
                       label="Fatal Recall KPI (0.92)")
            ax.set_ylim(0, 1.05); ax.set_ylabel("Score")
            ax.set_title("Fairness Audit 3 · Geographic Subgroup Performance", fontweight="bold")
            ax.legend(); plt.tight_layout(); plt.show()
    except Exception as _geo_err:
        print(f"Geographic audit skipped: {_geo_err}")

print("\n=== AI Ethics Audit Complete ===")
print("Summary:")
print("  1. Class imbalance disclosed — SMOTE applied to training set")
print("  2. Per-class recall measured — check Fatal Recall KPI (≥ 0.92)")
print("  3. Geographic parity gap measured — Urban vs Suburban/Rural")


In [ ]:
# ── Baseline results table + per-class report ─────────────────────────────────
baseline_df = pd.DataFrame([{k:v for k,v in r.items() if k!="_model"}
                             for r in baseline_results])
baseline_df = baseline_df.sort_values("MCC", ascending=False).reset_index(drop=True)

print("=" * 85)
print("BASELINE MODEL COMPARISON  (sorted by MCC)")
print("=" * 85)
print(baseline_df.to_string(index=False))

best_base = baseline_df.iloc[0]["Model"]
best_clf  = next(r["_model"] for r in baseline_results if r["Model"] == best_base)
print(f"\nBest baseline: {best_base}")
best_preds = best_clf.predict(X_test_sc)
print(classification_report(y_test, best_preds,
      target_names=["PD-Only (0)","Injury (1)","Fatal (2)"], zero_division=0))

plot_confusion_matrices_pair(
    y_test, best_preds, labels=["PD-Only", "Injury", "Fatal"],
    title_prefix=f"Best Baseline — {best_base}",
)

### Section 8.3 · Dynamic GridSearchCV

`StratifiedKFold(5)` ensures the rare Fatal class appears in every fold.
Scoring: `f1_macro` — penalises missed Fatal events as heavily as missed PD-Only.

**L1 (Lasso) vs L2 (Ridge) in Logistic Regression:**
- L1 zeroes out irrelevant features → automatic feature selection; sparse, interpretable model
- L2 shrinks all coefficients → smoother but all features retained
GridSearch will find the optimal trade-off.

In [ ]:
# ── 8.3  GridSearchCV – Hyperparameter Tuning ────────────────────────────────
#
# SPEED OPTIMISATIONS applied (vs. original):
#   • n_jobs=-1  on every GridSearchCV  → uses all CPU cores (3-6× faster)
#   • cv=3       instead of cv=5        → 40 % fewer fits, still statistically valid
#   • RF grid trimmed                   → removed n_estimators=300 (biggest time sink)
#   • RF itself  n_jobs=-1              → each forest built in parallel
#   • Search on 30 % stratified sample  → finds best params fast, then refits on full data
#
# Expected runtime on a typical laptop:  ~20-40 min  (vs. 7 h before)

import time

# ── 30 % stratified sample for the search phase ──────────────────────────────
from sklearn.model_selection import StratifiedShuffleSplit
_sss   = StratifiedShuffleSplit(n_splits=1, test_size=0.70, random_state=42)
_idx   = next(_sss.split(X_train_sc, y_train_sm))[0]
X_srch = X_train_sc[_idx]
y_srch = y_train_sm[_idx]
print(f"Search sample : {X_srch.shape[0]:,} rows  (30 % of {X_train_sc.shape[0]:,})")
print(f"Full train set: {X_train_sc.shape[0]:,} rows  (used for final refit)")

cv3 = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)

best_estimators = {}
t0 = time.time()

# ── Logistic Regression (L1 / L2) ────────────────────────────────────────────
print("\n[1/3] Logistic Regression GridSearch ...")
lr_grid = {
    "C"      : [0.01, 0.1, 1, 10],
    "penalty": ["l1", "l2"],
    "solver" : ["liblinear"],          # liblinear handles both L1 & L2
}
gs_lr = GridSearchCV(
    LogisticRegression(max_iter=1000, class_weight="balanced"),
    lr_grid, cv=cv3, scoring="f1_macro",
    n_jobs=-1, verbose=1              # n_jobs=-1 = all cores
)
gs_lr.fit(X_srch, y_srch)
# Refit best params on FULL training data
best_lr = LogisticRegression(max_iter=1000, class_weight="balanced",
                              **gs_lr.best_params_).fit(X_train_sc, y_train_sm)
best_estimators["Logistic Regression (Tuned)"] = best_lr
print(f"   Best params : {gs_lr.best_params_}")
print(f"   CV F1-macro : {gs_lr.best_score_:.4f}  ({time.time()-t0:.0f}s elapsed)")

# ── Random Forest ─────────────────────────────────────────────────────────────
print("\n[2/3] Random Forest GridSearch ...")
rf_grid = {
    "n_estimators"    : [100, 200],        # dropped 300 — not worth the time
    "max_depth"       : [None, 10, 20],
    "min_samples_split": [2, 5],
}
gs_rf = GridSearchCV(
    RandomForestClassifier(class_weight="balanced", n_jobs=-1, random_state=42),
    rf_grid, cv=cv3, scoring="f1_macro",
    n_jobs=-1, verbose=1
)
gs_rf.fit(X_srch, y_srch)
best_rf = RandomForestClassifier(class_weight="balanced", n_jobs=-1, random_state=42,
                                  **gs_rf.best_params_).fit(X_train_sc, y_train_sm)
best_estimators["Random Forest (Tuned)"] = best_rf
print(f"   Best params : {gs_rf.best_params_}")
print(f"   CV F1-macro : {gs_rf.best_score_:.4f}  ({time.time()-t0:.0f}s elapsed)")

# ── LightGBM ─────────────────────────────────────────────────────────────────
if LGBM_OK:
    print("\n[3/3] LightGBM GridSearch ...")
    lgbm_grid = {
        "n_estimators" : [200, 400],
        "learning_rate": [0.05, 0.1],
        "num_leaves"   : [31, 63],
    }
    gs_lgbm = GridSearchCV(
        lgb.LGBMClassifier(class_weight="balanced", random_state=42,
                            n_jobs=-1, verbose=-1),
        lgbm_grid, cv=cv3, scoring="f1_macro",
        n_jobs=-1, verbose=1
    )
    gs_lgbm.fit(X_srch, y_srch)
    best_lgbm = lgb.LGBMClassifier(class_weight="balanced", random_state=42,
                                    n_jobs=-1, verbose=-1,
                                    **gs_lgbm.best_params_).fit(X_train_sc, y_train_sm)
    best_estimators["LightGBM (Tuned)"] = best_lgbm
    print(f"   Best params : {gs_lgbm.best_params_}")
    print(f"   CV F1-macro : {gs_lgbm.best_score_:.4f}  ({time.time()-t0:.0f}s elapsed)")
else:
    print("\n[3/3] LightGBM skipped (not installed).")

total = time.time() - t0
print(f"\nGridSearchCV complete in {total/60:.1f} min  ({len(best_estimators)} models tuned)")
print("best_estimators dict keys:", list(best_estimators.keys()))


Search sample : 477,552 rows  (30 % of 1,591,842)
Full train set: 1,591,842 rows  (used for final refit)

[1/3] Logistic Regression GridSearch ...
Fitting 3 folds for each of 8 candidates, totalling 24 fits
   Best params : {'C': 0.01, 'penalty': 'l1', 'solver': 'liblinear'}
   CV F1-macro : 0.4801  (64s elapsed)

[2/3] Random Forest GridSearch ...
Fitting 3 folds for each of 12 candidates, totalling 36 fits
   Best params : {'max_depth': None, 'min_samples_split': 5, 'n_estimators': 100}
   CV F1-macro : 0.5641  (1476s elapsed)

[3/3] LightGBM GridSearch ...
Fitting 3 folds for each of 8 candidates, totalling 24 fits


In [ ]:
# ── 8.3b  Evaluate Tuned Models ──────────────────────────────────────────────
# Reads from best_estimators{} dict populated by the GridSearchCV cell above.

tuned_results = []
for name, clf in best_estimators.items():
    res = evaluate(name, clf, X_train_sc, y_train_sm, X_test_sc, y_test)
    tuned_results.append(res)
    print(f"{name:35s} Acc={res['Accuracy']}  "
          f"Rec(M)={res['Rec (M)']}  MCC={res['MCC']}  "
          f"AUC={res['AUC (OvR)']}  Time={res['Time (s)']}s")

cols = ["Model","Accuracy","Prec (M)","Rec (M)","F1 (M)","F1 (W)","MCC","AUC (OvR)","Time (s)"]
df_tuned = pd.DataFrame(tuned_results)[cols]
print("\n=== Tuned Model Results ===")
print(df_tuned.to_string(index=False))


Logistic Regression (Tuned)         Acc=0.7777  Rec(M)=0.5642  MCC=0.144  AUC=0.637  Time=11.5s
Random Forest (Tuned)               Acc=0.7407  Rec(M)=0.4813  MCC=0.1482  AUC=0.6092  Time=152.2s


### Section 8.3c · Confusion Matrices (All Tuned Models)

Each model shows **two** matrices side by side:
- **Without Normalization** — raw prediction counts
- **With Normalization** — row percentages (recall per true class; useful under class imbalance)


In [ ]:
# ── 8.3c  Confusion matrices for all tuned sklearn models ───────────────────
if "best_estimators" not in dir() or not best_estimators:
    print("Run Section 8.3 GridSearchCV first.")
else:
    for name, clf in best_estimators.items():
        preds = clf.predict(X_test_sc)
        print(f"\n{'='*60}\n  {name}\n{'='*60}")
        plot_confusion_matrices_pair(
            y_test, preds, labels=["PD-Only", "Injury", "Fatal"],
            title_prefix=name,
        )


### Variable Aliases — Backward Compatibility
These aliases ensure all downstream cells work regardless of which GridSearchCV
variable naming style they use (`rf_grid.best_estimator_` or `best_estimators[...]`).


In [ ]:
# ── Backward-compatibility aliases ───────────────────────────────────────────
# After this cell, BOTH styles work in every downstream cell:
#   rf_grid.best_estimator_                  (old style)
#   best_estimators["Random Forest (Tuned)"] (new style)

class _Alias:
    """Wraps a fitted model to expose .best_estimator_ attribute."""
    def __init__(self, model):
        self.best_estimator_ = model
        # Also expose common GridSearchCV attributes so old code doesn't break
        self.best_params_  = getattr(model, "get_params", lambda: {})()
        self.best_score_   = None

# Create aliases — safe even if a model wasn't trained (e.g. LGBM not installed)
lr_grid   = _Alias(best_estimators.get("Logistic Regression (Tuned)"))
rf_grid   = _Alias(best_estimators.get("Random Forest (Tuned)"))
lgbm_grid = _Alias(best_estimators.get("LightGBM (Tuned)")) if LGBM_OK else None

# Convenience: also expose individual best models as top-level variables
best_lr   = best_estimators.get("Logistic Regression (Tuned)")
best_rf   = best_estimators.get("Random Forest (Tuned)")
best_lgbm = best_estimators.get("LightGBM (Tuned)") if LGBM_OK else None

print("Aliases set:")
for name, obj in [("lr_grid", lr_grid), ("rf_grid", rf_grid), ("lgbm_grid", lgbm_grid)]:
    status = type(obj.best_estimator_).__name__ if obj and obj.best_estimator_ else "None"
    print(f"  {name:12s} .best_estimator_ = {status}")


### Section 8.4 · PyTorch Deep Neural Network

Architecture (Jiang et al., 2024 Table 6):
```
Input(8) → Dense(256)+BN+ReLU+Dropout(0.3)
         → Dense(128)+BN+ReLU+Dropout(0.3)
         → Dense(64)+BN+ReLU+Dropout(0.3)
         → Dense(3) → Softmax
```
Paper 2 result on SDOT+DfT: **Accuracy=91.12%, Recall=95.5%** — our benchmark target.

> If PyTorch could not load (OSError/WinError 182), `TORCH_OK=False` and this
> section is skipped cleanly. All sklearn models still run.

In [43]:
# Model 1: Simple DNN
class ShieldDNN(nn.Module):
            def __init__(self, n_in, n_out, p=0.3):
                super().__init__()
                self.net = nn.Sequential(
                    nn.Linear(n_in, 256), nn.BatchNorm1d(256), nn.ReLU(), nn.Dropout(p),
                    nn.Linear(256, 128),  nn.BatchNorm1d(128), nn.ReLU(), nn.Dropout(p),
                    nn.Linear(128, 64),   nn.BatchNorm1d(64),  nn.ReLU(), nn.Dropout(p),
                    nn.Linear(64, n_out)
                )
            def forward(self, x): return self.net(x)


In [ ]:
# ── 8.4 PyTorch DNN ──────────────────────────────────────────────────────────
# TORCH_OK validated in Cell 3 — catches corrupted installs (e.g. WinError 182)
dnn_result = None
device     = None

if not TORCH_OK:
    print("PyTorch unavailable (TORCH_OK=False). DNN section skipped.")
    print("All sklearn models are unaffected.")
else:
    try:
        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        print(f"PyTorch device : {device}")

        X_tr_t = torch.FloatTensor(X_train_sc).to(device)
        y_tr_t = torch.LongTensor(y_train_sm).to(device)
        X_te_t = torch.FloatTensor(X_test_sc).to(device)
        y_te_t = torch.LongTensor(y_test).to(device)

        cc = np.bincount(y_train_sm, minlength=3).astype(float)
        cw = torch.FloatTensor(1.0 / (cc + 1e-6)).to(device)
        cw = cw / cw.sum() * 3

        train_ds = TensorDataset(X_tr_t, y_tr_t)
        train_dl = DataLoader(train_ds, batch_size=256, shuffle=True)

        model_dnn = ShieldDNN(X_train_sc.shape[1], 3).to(device)
        criterion = nn.CrossEntropyLoss(weight=cw)
        optimizer = optim.Adam(model_dnn.parameters(), lr=1e-3, weight_decay=1e-4)
        scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, "min", patience=5, factor=0.5)

        EPOCHS, PATIENCE = 80, 10
        best_val, wait, best_state = float("inf"), 0, None
        t_losses, v_losses = [], []

        print(f"Training up to {EPOCHS} epochs (early stop patience={PATIENCE})...")
        for epoch in range(EPOCHS):
            model_dnn.train()
            ep_loss = 0.0
            for xb, yb in train_dl:
                optimizer.zero_grad()
                loss = criterion(model_dnn(xb), yb)
                loss.backward(); optimizer.step()
                ep_loss += loss.item() * len(xb)
            ep_loss /= len(train_ds)
            model_dnn.eval()
            with torch.no_grad():
                vl = criterion(model_dnn(X_te_t), y_te_t).item()
            scheduler.step(vl)
            t_losses.append(ep_loss); v_losses.append(vl)
            if vl < best_val:
                best_val   = vl
                best_state = {k: v.clone() for k, v in model_dnn.state_dict().items()}
                wait = 0
            else:
                wait += 1
                if wait >= PATIENCE:
                    print(f"  Early stop at epoch {epoch+1}")
                    break
            if (epoch + 1) % 10 == 0:
                print(f"  Ep {epoch+1:3d} | train={ep_loss:.4f} | val={vl:.4f}")

        model_dnn.load_state_dict(best_state)
        print("Best weights restored.")

        plt.figure(figsize=(9, 4))
        plt.plot(t_losses, label="Train loss", color="#4C72B0")
        plt.plot(v_losses, label="Val loss",   color="#DD8452")
        plt.title("Gated Linear Unit Training Curve", fontweight="bold")
        plt.xlabel("Epoch"); plt.ylabel("Loss"); plt.legend()
        plt.tight_layout(); plt.show()

        model_dnn.eval()
        with torch.no_grad():
            logits = model_dnn(X_te_t)
            probs  = torch.softmax(logits, dim=1).cpu().numpy()
            preds  = logits.argmax(dim=1).cpu().numpy()

        dnn_result_GLU = {
            "Model"    : "PyTorch DNN - Gated Linear Unit",
            "Accuracy" : round(accuracy_score(y_test, preds), 4),
            "Prec (M)" : round(precision_score(y_test, preds, average="macro", zero_division=0), 4),
            "Rec (M)"  : round(recall_score(y_test, preds, average="macro", zero_division=0), 4),
            "F1 (M)"   : round(f1_score(y_test, preds, average="macro", zero_division=0), 4),
            "F1 (W)"   : round(f1_score(y_test, preds, average="weighted", zero_division=0), 4),
            "MCC"      : round(matthews_corrcoef(y_test, preds), 4),
            "AUC (OvR)": round(roc_auc_score(y_test, probs, multi_class="ovr", average="macro"), 4),
            "_model"   : model_dnn,
        }
        print(f"Gated Linear Unit: Acc={dnn_result['Accuracy']}  Rec(M)={dnn_result['Rec (M)']}  "
              f"F1={dnn_result['F1 (M)']}  MCC={dnn_result['MCC']}")
        print(classification_report(y_test, preds,
              target_names=["PD-Only", "Injury", "Fatal"], zero_division=0))

        plot_confusion_matrices_pair(
            y_test, preds, labels=["PD-Only", "Injury", "Fatal"],
            title_prefix="PyTorch DNN",
            cmap="Purples",
        )

    except Exception as _dnn_err:
        print(f"DNN training failed ({type(_dnn_err).__name__}): {_dnn_err}")
        print("Skipping DNN — all sklearn models still run and comparison table will exclude DNN.")
        dnn_result_GLU = None

In [ ]:
# Model 2: Tabular ResNet
class ShieldDNN(nn.Module):
    # Tabular ResNet Implementation
    def __init__(self, n_in, n_out, p=0.2):
        super().__init__()
        self.input_layer = nn.Sequential(
            nn.Linear(n_in, 64),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.Dropout(p)
        )
        self.residual_block = nn.Sequential(
            nn.Linear(64, 64),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.Dropout(p)
        )
        self.output_layer = nn.Linear(64, n_out)

    def forward(self, x):
        x = self.input_layer(x)
        # Skip connection adds the input of the block to its output
        x = x + self.residual_block(x)
        return self.output_layer(x)

# # ── Default 1: right-sized architecture for ~8-17 tabular features (was 256/128/64).
    # class ShieldDNN(nn.Module):
    #     def __init__(self, n_in, n_out, p=0.2):
    #         super().__init__()
    #         self.net = nn.Sequential(
    #             nn.Linear(n_in, 64), nn.BatchNorm1d(64), nn.ReLU(), nn.Dropout(p),
    #             nn.Linear(64, 32),   nn.BatchNorm1d(32), nn.ReLU(), nn.Dropout(p),
    #             nn.Linear(32, n_out)
    #         )
    #     def forward(self, x): return self.net(x)

In [ ]:
# ── 8.4 PyTorch DNN ──────────────────────────────────────────────────────────
# TORCH_OK validated in Cell 3 — catches corrupted installs (e.g. WinError 182)
dnn_result = None
device     = None

if not TORCH_OK:
    print("PyTorch unavailable (TORCH_OK=False). DNN section skipped.")
    print("All sklearn models are unaffected.")
else:
    try:
        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        print(f"PyTorch device : {device}")

        X_tr_t = torch.FloatTensor(X_train_sc).to(device)
        y_tr_t = torch.LongTensor(y_train_sm).to(device)
        X_te_t = torch.FloatTensor(X_test_sc).to(device)
        y_te_t = torch.LongTensor(y_test).to(device)

        cc = np.bincount(y_train_sm, minlength=3).astype(float)
        cw = torch.FloatTensor(1.0 / (cc + 1e-6)).to(device)
        cw = cw / cw.sum() * 3

        train_ds = TensorDataset(X_tr_t, y_tr_t)
        train_dl = DataLoader(train_ds, batch_size=256, shuffle=True)

        model_dnn = ShieldDNN(X_train_sc.shape[1], 3).to(device)
        criterion = nn.CrossEntropyLoss(weight=cw)
        optimizer = optim.Adam(model_dnn.parameters(), lr=1e-3, weight_decay=1e-4)
        scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, "min", patience=5, factor=0.5)

        EPOCHS, PATIENCE = 80, 10
        best_val, wait, best_state = float("inf"), 0, None
        t_losses, v_losses = [], []

        print(f"Training up to {EPOCHS} epochs (early stop patience={PATIENCE})...")
        for epoch in range(EPOCHS):
            model_dnn.train()
            ep_loss = 0.0
            for xb, yb in train_dl:
                optimizer.zero_grad()
                loss = criterion(model_dnn(xb), yb)
                loss.backward(); optimizer.step()
                ep_loss += loss.item() * len(xb)
            ep_loss /= len(train_ds)
            model_dnn.eval()
            with torch.no_grad():
                vl = criterion(model_dnn(X_te_t), y_te_t).item()
            scheduler.step(vl)
            t_losses.append(ep_loss); v_losses.append(vl)
            if vl < best_val:
                best_val   = vl
                best_state = {k: v.clone() for k, v in model_dnn.state_dict().items()}
                wait = 0
            else:
                wait += 1
                if wait >= PATIENCE:
                    print(f"  Early stop at epoch {epoch+1}")
                    break
            if (epoch + 1) % 10 == 0:
                print(f"  Ep {epoch+1:3d} | train={ep_loss:.4f} | val={vl:.4f}")

        model_dnn.load_state_dict(best_state)
        print("Best weights restored.")

        plt.figure(figsize=(9, 4))
        plt.plot(t_losses, label="Train loss", color="#4C72B0")
        plt.plot(v_losses, label="Val loss",   color="#DD8452")
        plt.title("Gated Linear Unit Training Curve", fontweight="bold")
        plt.xlabel("Epoch"); plt.ylabel("Loss"); plt.legend()
        plt.tight_layout(); plt.show()

        model_dnn.eval()
        with torch.no_grad():
            logits = model_dnn(X_te_t)
            probs  = torch.softmax(logits, dim=1).cpu().numpy()
            preds  = logits.argmax(dim=1).cpu().numpy()

        dnn_result_GLU = {
            "Model"    : "PyTorch DNN - Gated Linear Unit",
            "Accuracy" : round(accuracy_score(y_test, preds), 4),
            "Prec (M)" : round(precision_score(y_test, preds, average="macro", zero_division=0), 4),
            "Rec (M)"  : round(recall_score(y_test, preds, average="macro", zero_division=0), 4),
            "F1 (M)"   : round(f1_score(y_test, preds, average="macro", zero_division=0), 4),
            "F1 (W)"   : round(f1_score(y_test, preds, average="weighted", zero_division=0), 4),
            "MCC"      : round(matthews_corrcoef(y_test, preds), 4),
            "AUC (OvR)": round(roc_auc_score(y_test, probs, multi_class="ovr", average="macro"), 4),
            "_model"   : model_dnn,
        }
        print(f"Gated Linear Unit: Acc={dnn_result['Accuracy']}  Rec(M)={dnn_result['Rec (M)']}  "
              f"F1={dnn_result['F1 (M)']}  MCC={dnn_result['MCC']}")
        print(classification_report(y_test, preds,
              target_names=["PD-Only", "Injury", "Fatal"], zero_division=0))

        plot_confusion_matrices_pair(
            y_test, preds, labels=["PD-Only", "Injury", "Fatal"],
            title_prefix="PyTorch DNN",
            cmap="Purples",
        )

    except Exception as _dnn_err:
        print(f"DNN training failed ({type(_dnn_err).__name__}): {_dnn_err}")
        print("Skipping DNN — all sklearn models still run and comparison table will exclude DNN.")
        dnn_result_GLU = None

In [ ]:
# Model 3:Gated Linear Unit
class ShieldDNN(nn.Module):
    # Gated Linear Unit Implementation
    def __init__(self, n_in, n_out, p=0.2):
        super().__init__()
        # Double the output size to accommodate the gate splitting
        self.fc = nn.Linear(n_in, 128)
        self.bn = nn.BatchNorm1d(64)
        self.drop = nn.Dropout(p)
        self.output_layer = nn.Linear(64, n_out)

    def forward(self, x):
        x = self.fc(x)
        # Split the tensor into two equal halves along the last dimension
        x, gate = x.chunk(2, dim=-1)
        # Apply the gating mechanism
        x = x * torch.sigmoid(gate)
        x = self.bn(x)
        x = self.drop(x)
        return self.output_layer(x)

In [ ]:
# ── 8.4 PyTorch DNN ──────────────────────────────────────────────────────────
# TORCH_OK validated in Cell 3 — catches corrupted installs (e.g. WinError 182)
dnn_result = None
device     = None

if not TORCH_OK:
    print("PyTorch unavailable (TORCH_OK=False). DNN section skipped.")
    print("All sklearn models are unaffected.")
else:
    try:
        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        print(f"PyTorch device : {device}")

        X_tr_t = torch.FloatTensor(X_train_sc).to(device)
        y_tr_t = torch.LongTensor(y_train_sm).to(device)
        X_te_t = torch.FloatTensor(X_test_sc).to(device)
        y_te_t = torch.LongTensor(y_test).to(device)

        cc = np.bincount(y_train_sm, minlength=3).astype(float)
        cw = torch.FloatTensor(1.0 / (cc + 1e-6)).to(device)
        cw = cw / cw.sum() * 3

        train_ds = TensorDataset(X_tr_t, y_tr_t)
        train_dl = DataLoader(train_ds, batch_size=256, shuffle=True)

        model_dnn = ShieldDNN(X_train_sc.shape[1], 3).to(device)
        criterion = nn.CrossEntropyLoss(weight=cw)
        optimizer = optim.Adam(model_dnn.parameters(), lr=1e-3, weight_decay=1e-4)
        scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, "min", patience=5, factor=0.5)

        EPOCHS, PATIENCE = 80, 10
        best_val, wait, best_state = float("inf"), 0, None
        t_losses, v_losses = [], []

        print(f"Training up to {EPOCHS} epochs (early stop patience={PATIENCE})...")
        for epoch in range(EPOCHS):
            model_dnn.train()
            ep_loss = 0.0
            for xb, yb in train_dl:
                optimizer.zero_grad()
                loss = criterion(model_dnn(xb), yb)
                loss.backward(); optimizer.step()
                ep_loss += loss.item() * len(xb)
            ep_loss /= len(train_ds)
            model_dnn.eval()
            with torch.no_grad():
                vl = criterion(model_dnn(X_te_t), y_te_t).item()
            scheduler.step(vl)
            t_losses.append(ep_loss); v_losses.append(vl)
            if vl < best_val:
                best_val   = vl
                best_state = {k: v.clone() for k, v in model_dnn.state_dict().items()}
                wait = 0
            else:
                wait += 1
                if wait >= PATIENCE:
                    print(f"  Early stop at epoch {epoch+1}")
                    break
            if (epoch + 1) % 10 == 0:
                print(f"  Ep {epoch+1:3d} | train={ep_loss:.4f} | val={vl:.4f}")

        model_dnn.load_state_dict(best_state)
        print("Best weights restored.")

        plt.figure(figsize=(9, 4))
        plt.plot(t_losses, label="Train loss", color="#4C72B0")
        plt.plot(v_losses, label="Val loss",   color="#DD8452")
        plt.title("Gated Linear Unit Training Curve", fontweight="bold")
        plt.xlabel("Epoch"); plt.ylabel("Loss"); plt.legend()
        plt.tight_layout(); plt.show()

        model_dnn.eval()
        with torch.no_grad():
            logits = model_dnn(X_te_t)
            probs  = torch.softmax(logits, dim=1).cpu().numpy()
            preds  = logits.argmax(dim=1).cpu().numpy()

        dnn_result_GLU = {
            "Model"    : "PyTorch DNN - Gated Linear Unit",
            "Accuracy" : round(accuracy_score(y_test, preds), 4),
            "Prec (M)" : round(precision_score(y_test, preds, average="macro", zero_division=0), 4),
            "Rec (M)"  : round(recall_score(y_test, preds, average="macro", zero_division=0), 4),
            "F1 (M)"   : round(f1_score(y_test, preds, average="macro", zero_division=0), 4),
            "F1 (W)"   : round(f1_score(y_test, preds, average="weighted", zero_division=0), 4),
            "MCC"      : round(matthews_corrcoef(y_test, preds), 4),
            "AUC (OvR)": round(roc_auc_score(y_test, probs, multi_class="ovr", average="macro"), 4),
            "_model"   : model_dnn,
        }
        print(f"Gated Linear Unit: Acc={dnn_result['Accuracy']}  Rec(M)={dnn_result['Rec (M)']}  "
              f"F1={dnn_result['F1 (M)']}  MCC={dnn_result['MCC']}")
        print(classification_report(y_test, preds,
              target_names=["PD-Only", "Injury", "Fatal"], zero_division=0))

        plot_confusion_matrices_pair(
            y_test, preds, labels=["PD-Only", "Injury", "Fatal"],
            title_prefix="PyTorch DNN",
            cmap="Purples",
        )

    except Exception as _dnn_err:
        print(f"DNN training failed ({type(_dnn_err).__name__}): {_dnn_err}")
        print("Skipping DNN — all sklearn models still run and comparison table will exclude DNN.")
        dnn_result_GLU = None

### Section 8.5 · Head-to-Head Model Comparison

**Metric ranking (most important → least):**
1. **Macro Recall** — catches Fatal events; asymmetric error cost
2. **MCC** — most reliable single metric under class imbalance
3. **AUC** — threshold-independent discrimination
4. **Macro F1** — balanced precision-recall
5. **Accuracy** — least informative given imbalance

**Paper 2 benchmark (red dashed line):** RF accuracy = 0.878

In [ ]:
# ── Full comparison table ─────────────────────────────────────────────────────
all_results = baseline_results + tuned_results
if dnn_result:
    all_results.append(dnn_result)
if dnn_1_result:
    all_results.append(dnn_1_result)
if dnn_2_result:
    all_results.append(dnn_2_result)


comp_df = pd.DataFrame([{k:v for k,v in r.items() if k!="_model"}
                         for r in all_results])
comp_df = comp_df.sort_values("MCC", ascending=False).reset_index(drop=True)

print("=" * 95)
print("FULL MODEL COMPARISON  (sorted by MCC)")
print("=" * 95)
print(comp_df.to_string(index=False))

# Grouped bar chart
metrics = ["Accuracy","Prec (M)","Rec (M)","F1 (M)","MCC","AUC (OvR)"]
x = np.arange(len(comp_df)); width = 0.13
fig, ax = plt.subplots(figsize=(16, 6))
colors = ["#4C72B0","#DD8452","#55A868","#C44E52","#8172B2","#937860"]
for i, metric in enumerate(metrics):
    ax.bar(x + i*width, comp_df[metric].fillna(0).values, width,
           label=metric, color=colors[i], alpha=0.85)
ax.axhline(0.878, color="red", linestyle="--", linewidth=1.5, alpha=0.8,
           label="Paper 2 RF benchmark (0.878)")
ax.set_xticks(x + width*(len(metrics)-1)/2)
ax.set_xticklabels(comp_df["Model"], rotation=20, ha="right", fontsize=9)
ax.set_ylim(0, 1.1); ax.set_ylabel("Score")
ax.set_title("Model Performance Comparison - Ontario Smart-Shield",
             fontweight="bold", fontsize=12)
ax.legend(loc="upper right", fontsize=8)
plt.tight_layout(); plt.show()

# ROC curves per class for top models
y_bin = label_binarize(y_test, classes=[0,1,2])
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle("ROC Curves per Class - Top Models", fontweight="bold")
class_names = ["PD-Only","Injury","Fatal"]

roc_models = {"RF (tuned)": best_estimators["Random Forest (Tuned)"],
              "LR (tuned)": best_estimators["Logistic Regression (Tuned)"]}
if LGBM_OK and "LightGBM (Tuned)" in best_estimators: roc_models["LightGBM (tuned)"] = best_estimators["LightGBM (Tuned)"]

for ax, cls_idx in zip(axes, range(3)):
    for name, clf in roc_models.items():
        try:
            fp, tp, _ = roc_curve(y_bin[:, cls_idx], clf.predict_proba(X_test_sc)[:, cls_idx])
            auc_cls   = roc_auc_score(y_bin[:, cls_idx], clf.predict_proba(X_test_sc)[:, cls_idx])
            ax.plot(fp, tp, label=f"{name} ({auc_cls:.3f})", lw=1.5)
        except Exception:
            pass
    if dnn_result and TORCH_OK:
        fp, tp, _ = roc_curve(y_bin[:, cls_idx], probs[:, cls_idx])
        auc_cls   = roc_auc_score(y_bin[:, cls_idx], probs[:, cls_idx])
        ax.plot(fp, tp, label=f"DNN ({auc_cls:.3f})", lw=2, linestyle="--")
    ax.plot([0,1],[0,1],"k--", lw=0.8)
    ax.set_title(f"Class: {class_names[cls_idx]}"); ax.set_xlabel("FPR"); ax.set_ylabel("TPR")
    ax.legend(fontsize=7)
plt.tight_layout(); plt.show()

### Section 8.6 · Live Test Cases – Ontario Highway Scenarios

Five realistic Ontario scenarios stress-test boundary conditions.
TC-2 and TC-5 **must** be classified as Fatal-risk (class 2) for the model to
meet the project KPI of Recall ≥ 0.92 on fatal events.

| # | Scenario | Expected | Key risk factors |
|---|---|---|---|
| TC-1 | Clear summer afternoon, 401 rush hour | Injury (1) | IS_RUSHHOUR=1 |
| TC-2 | Blizzard at 2am, Hwy 400, pedestrian struck | **Fatal (2)** | IS_NIGHT=1, PED=1, Jan |
| TC-3 | Wet dawn, bicycle involved, off-rush | Injury (1) | BICYCLE=1, Apr |
| TC-4 | Clear Sunday morning, Hwy 115 | PD-Only (0) | Low-risk profile |
| TC-5 | Ice storm rush hour, QEW, Feb 5pm | **Fatal (2)** | IS_RUSHHOUR=1, Feb |

In [ ]:
# ── Ontario live test cases ───────────────────────────────────────────────────
# Feature order: OCC_HOUR, MONTH_NUM, SEASON_NUM, IS_NIGHT, IS_RUSHHOUR,
#                PEDESTRIAN_BIN, BICYCLE_BIN, AUTOMOBILE_BIN
TC = {
    "TC-1 Clear rush-hour (401 Jul 5pm)" : [17, 7, 3, 0, 1, 0, 0, 1],
    "TC-2 Blizzard night (Hwy400 Jan 2am)": [ 2, 1, 1, 1, 0, 1, 0, 1],
    "TC-3 Wet dawn bicycle (Hwy7 Apr 6am)": [ 6, 4, 2, 0, 0, 0, 1, 1],
    "TC-4 Clear Sunday (Hwy115 Jun 9am)"  : [ 9, 6, 3, 0, 0, 0, 0, 1],
    "TC-5 Ice storm rush (QEW Feb 5pm)"   : [17, 2, 1, 0, 1, 0, 0, 1],
}
EXPECTED = [1, 2, 1, 0, 2]
CLASS_LABELS = {0:"PD-Only", 1:"Injury", 2:"Fatal"}

tc_scaled = scaler.transform(np.array(list(TC.values()), dtype=float))

pred_models = {"RF (tuned)": best_estimators["Random Forest (Tuned)"],
               "LR (tuned)": best_estimators["Logistic Regression (Tuned)"]}
if LGBM_OK and "LightGBM (Tuned)" in best_estimators: pred_models["LightGBM (tuned)"] = best_estimators["LightGBM (Tuned)"]

header = f"{'Scenario':<42} {'Expected':<10}"
for m in pred_models: header += f"{m[:14]:<18}"
if TORCH_OK: header += "DNN"
print(header)
print("-" * (52 + 18*len(pred_models) + (6 if TORCH_OK else 0)))

for idx, (scenario, _) in enumerate(TC.items()):
    exp = CLASS_LABELS[EXPECTED[idx]]
    row = f"{scenario:<42} {exp:<10}"
    for name, clf in pred_models.items():
        p    = clf.predict(tc_scaled[idx:idx+1])[0]
        conf = clf.predict_proba(tc_scaled[idx:idx+1])[0][p] * 100
        tick = "OK" if p == EXPECTED[idx] else "XX"
        row += f"{CLASS_LABELS[p]}({conf:.0f}%){tick:<5}"
    if TORCH_OK:
        with torch.no_grad():
            lgt = model_dnn(torch.FloatTensor(tc_scaled[idx:idx+1]).to(device))
            pp  = lgt.argmax().item()
            cf  = torch.softmax(lgt,1)[0][pp].item()*100
        tick = "OK" if pp == EXPECTED[idx] else "XX"
        row += f"{CLASS_LABELS[pp]}({cf:.0f}%){tick}"
    print(row)

# Safety Score for each test case
print("\n--- Safety Score (S) per test case ---")
for scenario, feat in TC.items():
    occ_hour, month_num, season_num, is_night, is_rush, ped, bike, auto = feat
    surface_risk = 0.35 * (1.0 if season_num == 1 else 0.2)
    wind_risk    = 0.20 * (1.0 if season_num == 1 else 0.1)
    visibility   = 0.30 * (0.8 * is_night + 0.1 * (1 - is_night))
    temp_risk    = 0.15 * (1.0 if month_num in [12,1,2] else 0.1)
    E_index      = min(1.0, surface_risk + wind_risk + visibility + temp_risk)
    T_score      = 0.5 * is_rush
    V_score      = 0.8 if (ped or bike) else 0.2
    S            = (0.25*T_score + 0.35*V_score + 0.40*E_index) * 100
    tier = "HIGH" if S>=71 else ("MEDIUM" if S>=31 else "LOW")
    print(f"  {scenario:<42}  S={S:5.1f}  [{tier}]")

### Section 8.7 · Final Model Selection & Rationale

**Decision matrix (weighted scoring):**

| Criterion | Weight | LR L1 | RF (tuned) | LightGBM | DNN |
|---|---|---|---|---|---|
| Macro Recall (Fatal) | 40% | Low | Medium | Medium | **High** |
| MCC | 25% | Low | **High** | High | High |
| Interpretability | 20% | **Yes** | Partial | Partial | No |
| Inference speed | 15% | **Fast** | Medium | Fast | Slow |

**Final decisions:**
- **Deploy**: Random Forest (tuned) — best MCC, partial interpretability, Paper 2 benchmark
- **Oracle**: PyTorch DNN — use when RF confidence < 60% on high-risk prediction
- **Audit/Report**: Logistic Regression L1 — fully auditable coefficients for D3 deliverable

In [ ]:
# ── Final model evaluation ────────────────────────────────────────────────────
final_model = best_estimators["Random Forest (Tuned)"]
final_preds = final_model.predict(X_test_sc)
final_probs = final_model.predict_proba(X_test_sc)

print("=" * 60)
print("FINAL MODEL: Random Forest (GridSearch Tuned)")
print("=" * 60)
bp = gs_rf.best_params_ if "gs_rf" in dir() else final_model.get_params()
print(f"Best params: {bp}\n")
print(classification_report(y_test, final_preds,
      target_names=["PD-Only (0)","Injury (1)","Fatal (2)"], zero_division=0))

fi_final = pd.Series(final_model.feature_importances_, index=available).sort_values()
fig, ax = plt.subplots(figsize=(8, 5))

fi_final.plot(kind="barh", ax=ax, color="#4C72B0")
ax.set_title("Feature Importances - Final RF Model", fontweight="bold")
ax.set_xlabel("Mean Decrease in Impurity")
for bar, val in zip(ax.patches, fi_final.values):
    ax.text(val+0.001, bar.get_y()+bar.get_height()/2,
                 f"{val:.3f}", va="center", fontsize=9)

plt.tight_layout(); plt.show()

plot_confusion_matrices_pair(
    y_test, final_preds, labels=["PD-Only", "Injury", "Fatal"],
    title_prefix="Final Model — Random Forest (Tuned)",
)

# Comparison vs Paper 2
paper2_rf = {"Accuracy":0.8782, "Rec (M)":0.8782, "F1 (M)":0.8780, "AUC (OvR)":0.8520}
our_rf    = {"Accuracy": accuracy_score(y_test, final_preds),
             "Rec (M)":  recall_score(y_test, final_preds, average="macro", zero_division=0),
             "F1 (M)":   f1_score(y_test, final_preds, average="macro", zero_division=0),
             "AUC (OvR)":roc_auc_score(y_test, final_probs, multi_class="ovr", average="macro")}
print("\n--- vs. Jiang et al. (2024) Random Forest ---")
print(f"{'Metric':<15} {'Paper 2':>10}  {'Ours':>10}  {'Delta':>10}")
print("-" * 50)
for m in paper2_rf:
    delta = our_rf[m] - paper2_rf[m]
    print(f"{m:<15} {paper2_rf[m]:>10.4f}  {our_rf[m]:>10.4f}  {delta:>+10.4f}")

---
## Section 9 · Summary & Sprint Progress

### Completed in this notebook

| Sprint | Deliverable | Section |
|--------|-------------|---------|
| Sprint 1–2 | EDA, stats, preprocessing, feature selection | 1–5 |
| Sprint 2 | Baselines, GridSearchCV, DNN, comparison | 8 |
| Sprint 2 | Vision Brain sample images + CNN fine-tune | 6 |
| Sprint 2 | Ethics audit + confusion matrices | 7, 8.3c |
| **Sprint 3** | NLP Brain TF-IDF | **10.1** |
| **Sprint 3** | Safety Score fusion + dashboard | **10.2** |
| **Sprint 3** | SHAP explainability | **10.3** |
| **Sprint 3** | Model deployment (joblib) | **10.4** |

### Future work (Sprint 4+)
- Live Ontario 511 API feed (replace sample alerts)
- Production dashboard (Streamlit / Flask)
- Real-time camera frame ingestion from 511on.ca


---
# Section 10 · Sprint 3 — Multimodal Fusion & Deployment

Sprint 3 completes the **3-Brain architecture**:

| Pillar | Module | Output |
|--------|--------|--------|
| **1 NLP Brain** | TF-IDF on Ontario 511 alerts | `T` score (text hazard) |
| **2 Vision Brain** | ResNet18 (Section 6) | `V` score (snow/ice probability) |
| **3 Logistic Optimizer** | Tuned RF (Section 8) | Severity classification |
| **Fusion** | Safety Score formula | `S` → speed recommendation |

Also: **SHAP explainability** + **model serialization** for deployment.


In [ ]:
# ── PART SYNC → main notebook ──
import subprocess
import sys
from pathlib import Path

def _find_root() -> Path:
    for base in [Path.cwd(), *Path.cwd().parents]:
        if (base / "scripts" / "sync_notebook_parts.py").is_file():
            return base
    return Path.cwd()

_ROOT = _find_root()
rc = subprocess.run(
    [sys.executable, str(_ROOT / "scripts" / "sync_notebook_parts.py"),
     "--direction", "parts-to-main", "--only", "03_tabular_ml.ipynb"],
    cwd=_ROOT,
).returncode
print("Synced to capstone_with_results.ipynb" if rc == 0 else f"Sync failed (exit {rc})")